In [1]:
import dashscope
import os
import json
from pathlib import Path
import cv2

# API-Setup
dashscope.base_http_api_url = 'https://dashscope-intl.aliyuncs.com/api/v1'
api_key = os.getenv("QWEN_API_KEY")
if not api_key:
    raise RuntimeError("QWEN_API_KEY Umgebungsvariable nicht gesetzt.")

# Pfad zu den Videos
video_ordner = Path(r"C:\Users\miria\Deepfake_Detection\data\processed")
output_datei = Path(r"C:\Users\miria\Deepfake_Detection\qwen3-vl-235b-a22b-instruct_lang.json")

alle_ergebnisse = []

# Analysiere alle MP4-Dateien im Ordner (rekursiv in Unterordnern)
for video_pfad in video_ordner.glob("**/*.mp4"):
    print(f"\n--- Analysiere: {video_pfad.name} ---")
    
    # Video-Überprüfung
    cap = cv2.VideoCapture(str(video_pfad))
    if not cap.isOpened():
        print("Fehler: Video konnte nicht geladen werden.")
        cap.release()
        continue
    frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    print(f"Video geladen: {frames} Frames bei {fps} FPS")
    cap.release()
    
    video_id = video_pfad.stem  # Extrahiere die Video-ID hier

    try:
        # Prompt für die API – jetzt mit korrekter video_id-Interpolation
        prompt_text = f"""<role>
You are an expert in deepfake detection with specialization in multimodal forensics.
</role>

<task>
Analyze the provided video and evaluate whether it is real or fake.
Your analysis must be strictly based on the indicators listed below.
</task>

<indicators>
1. Unnatural body movements, stiff or robotic gestures, or inconsistent/missing facial expressions.
2. Facial irregularities: uneven skin tones, poor eye rendering, asymmetry, unnatural textures.
3. Inconsistencies in lighting, shadows, colors, reflections.
4. Signs of face swapping: unnatural lip-sync, distorted mouth shapes, abnormal eye movements or blinking.
5. Visual artifacts: blurring, pixel noise, edge glitches, compression issues, resolution inconsistencies, generator- or frequency-pattern anomalies.
6. Temporal instabilities: warping, jitter, frame jumps, changes in details across frames.
7. Logical coherence and contextual consistency of the visual content.
</indicators>

<output_format>
Return the result as a valid JSON object with exactly the following fields:
{{
  "video_id": "{video_id}",
  "assessment": "<Real or Fake>",
  "probability_fake": <integer 0-100>,
  "justification": "<max. 5 bullet points>"
}}
</output_format>

<constraints>
- Do NOT output explanations outside the JSON.
- Keep the justification concise, factual, and forensic.
</constraints>"""

        messages = [
            {
                "role": "user",
                "content": [
                    {"video": str(video_pfad), "fps": 5},
                    {"text": prompt_text}
                ]
            }
        ]
        
        # API-Aufruf
        response = dashscope.MultiModalConversation.call(
            api_key=api_key,
            # model='qwen3-vl-32b-instruct',
            model='qwen3-vl-235b-a22b-instruct',
            messages=messages
        )
        
        response_text = response.output.choices[0].message.content[0]["text"]
        print(f"API-Antwort: {response_text}")
        
        # JSON parsen (Markdown-Codeblock bereinigen)
        if response_text.startswith("```"):
            response_text = response_text.split("```")[1].strip()
            if response_text.startswith("json"):
                response_text = response_text[4:].strip()
        
        result = json.loads(response_text)
        alle_ergebnisse.append(result)
        print("✓ Erfolgreich geparst")
        
    except Exception as e:
        print(f"Fehler bei {video_pfad.name}: {e}")
        continue

# Ergebnisse speichern
with open(output_datei, 'w', encoding='utf-8') as f:
    json.dump(alle_ergebnisse, f, indent=2, ensure_ascii=False)

print(f"\nErgebnisse gespeichert in: {output_datei}")


--- Analysiere: 1_00069_8.mp4 ---
Video geladen: 241 Frames bei 24.0 FPS
API-Antwort: {
  "video_id": "1_00069_8",
  "assessment": "Real",
  "probability_fake": 5,
  "justification": [
    "Natural, fluid body movements with expressive hand gestures and head tilts consistent with live performance.",
    "Facial expressions are dynamic and symmetrical; skin texture and eye rendering appear natural without distortion.",
    "Lighting is consistent across frames with realistic shadows and reflections on the suit and background.",
    "Lip-sync matches visible mouth movements; no signs of face swapping or unnatural lip/eye behavior.",
    "No visual artifacts, edge glitches, or temporal instabilities observed; resolution and frame coherence are stable."
  ]
}
✓ Erfolgreich geparst

--- Analysiere: 1_00079_9.mp4 ---
Video geladen: 300 Frames bei 30.0 FPS
API-Antwort: {
  "video_id": "1_00079_9",
  "assessment": "Real",
  "probability_fake": 5,
  "justification": [
    "Natural hand gesture

In [2]:
import dashscope
import os
import json
from pathlib import Path
import cv2

# API-Setup
dashscope.base_http_api_url = 'https://dashscope-intl.aliyuncs.com/api/v1'
api_key = os.getenv("QWEN_API_KEY")
if not api_key:
    raise RuntimeError("QWEN_API_KEY Umgebungsvariable nicht gesetzt.")

# Pfad zu den Videos
video_ordner = Path(r"C:\Users\miria\Deepfake_Detection\data\processed")
output_datei = Path(r"C:\Users\miria\Deepfake_Detection\qwen3-vl-235b-a22b-instruct_lang.json")

# 1. Bestehende Ergebnisse laden, falls vorhanden
alle_ergebnisse = []
verarbeitete_ids = set()

if output_datei.exists():
    try:
        with open(output_datei, 'r', encoding='utf-8') as f:
            content = f.read()
            if content: # Prüfen, ob Datei nicht leer ist
                alle_ergebnisse = json.loads(content)
                # Erstelle ein Set der IDs für schnelles Nachschlagen
                verarbeitete_ids = {entry.get("video_id") for entry in alle_ergebnisse}
                print(f"Bereits analysierte Videos geladen: {len(alle_ergebnisse)}")
    except json.JSONDecodeError:
        print("Warnung: Ausgabedatei war beschädigt oder leer. Starte neu.")
        alle_ergebnisse = []

# Analysiere alle MP4-Dateien im Ordner (rekursiv in Unterordnern)
for video_pfad in video_ordner.glob("**/*.mp4"):
    video_id = video_pfad.stem  # Extrahiere die Video-ID (Dateiname ohne Endung)

    # 2. Prüfen, ob Video schon analysiert wurde
    if video_id in verarbeitete_ids:
        print(f"Überspringe (bereits vorhanden): {video_pfad.name}")
        continue

    print(f"\n--- Analysiere: {video_pfad.name} ---")
    
    # Video-Überprüfung
    cap = cv2.VideoCapture(str(video_pfad))
    if not cap.isOpened():
        print("Fehler: Video konnte nicht geladen werden.")
        cap.release()
        continue
    frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    print(f"Video geladen: {frames} Frames bei {fps} FPS")
    cap.release()
    
    try:
        # Prompt für die API
        prompt_text = f"""<role>
You are an expert in deepfake detection with specialization in multimodal forensics.
</role>

<task>
Analyze the provided video and evaluate whether it is real or fake.
Your analysis must be strictly based on the indicators listed below.
</task>

<indicators>
1. Unnatural body movements, stiff or robotic gestures, or inconsistent/missing facial expressions.
2. Facial irregularities: uneven skin tones, poor eye rendering, asymmetry, unnatural textures.
3. Inconsistencies in lighting, shadows, colors, reflections.
4. Signs of face swapping: unnatural lip-sync, distorted mouth shapes, abnormal eye movements or blinking.
5. Visual artifacts: blurring, pixel noise, edge glitches, compression issues, resolution inconsistencies, generator- or frequency-pattern anomalies.
6. Temporal instabilities: warping, jitter, frame jumps, changes in details across frames.
7. Logical coherence and contextual consistency of the visual content.
</indicators>

<output_format>
Return the result as a valid JSON object with exactly the following fields:
{{
  "video_id": "{video_id}",
  "assessment": "<Real or Fake>",
  "probability_fake": <integer 0-100>,
  "justification": "<max. 5 bullet points>"
}}
</output_format>

<constraints>
- Do NOT output explanations outside the JSON.
- Keep the justification concise, factual, and forensic.
</constraints>"""

        messages = [
            {
                "role": "user",
                "content": [
                    {"video": str(video_pfad), "fps": 5},
                    {"text": prompt_text}
                ]
            }
        ]
        
        # API-Aufruf
        response = dashscope.MultiModalConversation.call(
            api_key=api_key,
            # model='qwen3-vl-32b-instruct',
            model='qwen3-vl-235b-a22b-instruct',
            messages=messages
        )
        
        if response.status_code == 200:
            response_text = response.output.choices[0].message.content[0]["text"]
            print(f"API-Antwort erhalten.")
            
            # JSON parsen (Markdown-Codeblock bereinigen)
            if response_text.startswith("```"):
                response_text = response_text.split("```")[1].strip()
                if response_text.startswith("json"):
                    response_text = response_text[4:].strip()
            
            result = json.loads(response_text)
            
            # Ergebnis zur Liste hinzufügen
            alle_ergebnisse.append(result)
            verarbeitete_ids.add(video_id) # ID zum Set hinzufügen, falls im gleichen Lauf nochmal vorkommt
            print("✓ Erfolgreich geparst und zur Liste hinzugefügt")

            # 3. Speichern nach jedem erfolgreichen Aufruf (Sicherheit gegen Abstürze)
            with open(output_datei, 'w', encoding='utf-8') as f:
                json.dump(alle_ergebnisse, f, indent=2, ensure_ascii=False)
                
        else:
            print(f"API Fehler: {response.code} - {response.message}")

    except Exception as e:
        print(f"Fehler bei {video_pfad.name}: {e}")
        continue

print(f"\nAlle Analysen abgeschlossen. Ergebnisse in: {output_datei}")

Bereits analysierte Videos geladen: 48
Überspringe (bereits vorhanden): 1_00069_8.mp4
Überspringe (bereits vorhanden): 1_00079_9.mp4
Überspringe (bereits vorhanden): 1_00084_7.mp4
Überspringe (bereits vorhanden): 1_00135_9.mp4
Überspringe (bereits vorhanden): 1_00203.mp4
Überspringe (bereits vorhanden): 2_112_9.mp4
Überspringe (bereits vorhanden): 2_160_7.mp4
Überspringe (bereits vorhanden): 2_180_8.mp4
Überspringe (bereits vorhanden): 2_342.mp4
Überspringe (bereits vorhanden): 2_539.mp4
Überspringe (bereits vorhanden): 1_00261.mp4
Überspringe (bereits vorhanden): 1_00266_9.mp4
Überspringe (bereits vorhanden): 1_00267.mp4
Überspringe (bereits vorhanden): 1_00268.mp4
Überspringe (bereits vorhanden): 1_00269.mp4
Überspringe (bereits vorhanden): 1_00271.mp4

--- Analysiere: 1_00289.mp4 ---
Video geladen: 313 Frames bei 30.0 FPS
API-Antwort erhalten.
✓ Erfolgreich geparst und zur Liste hinzugefügt
Überspringe (bereits vorhanden): 1_id0_0000_7.mp4
Überspringe (bereits vorhanden): 1_id7_id12